# Physics-Informed Graph Neural Network (PIGNN) for BlueROV2 Dynamics

This notebook runs the full pipeline:
1. **Setup** — Install dependencies, configure paths
2. **Data Generation** — Simulate BlueROV2 trajectories
3. **Graph Inspection** — Visualise the heterogeneous graph structure
4. **Training** — Train the PIGNN with physics-informed losses
5. **Evaluation** — Rollout predictions vs ground truth

## 1. Setup

In [ ]:
# Uncomment and run once to install dependencies:
# !pip install torch torch-geometric numpy scipy matplotlib tqdm control numba tensorboard

In [ ]:
import os
import sys
import time

import numpy as np
import torch
import matplotlib.pyplot as plt
from IPython.display import display, HTML

# Make sure the project root is on the path
PROJECT_ROOT = os.path.abspath(".")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

print(f"Project root: {PROJECT_ROOT}")
print(f"PyTorch:      {torch.__version__}")
print(f"CUDA:         {torch.cuda.is_available()}")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device:       {device}")

## 2. Data Generation

Simulate BlueROV2 trajectories using the 4-DOF Fossen model. This creates training, dev, and test sets.

In [ ]:
import control as ct
from src.bluerov import bluerov
from data.data_utility import random_input, random_x0

np.random.seed(0)

rov_sys = ct.NonlinearIOSystem(
    bluerov, None,
    inputs=("X", "Y", "Z", "M_z"),
    outputs=("x", "y", "z", "psi", "u", "v", "w", "r"),
    states=("x", "y", "z", "psi", "u", "v", "w", "r"),
    name="bluerov_system",
)


def generate_dataset(name, n_traj, input_type, dt, T_tot, intervals, params=None):
    """Generate and save a trajectory dataset."""
    N = int(T_tot / dt)
    t = np.linspace(0, T_tot, N, dtype=np.float32)
    N_x, N_u = 9, 4

    X = np.zeros((n_traj, N, N_x), dtype=np.float32)
    U = np.zeros((n_traj, N, N_u), dtype=np.float32)
    t_coll = np.zeros((n_traj, N, 1), dtype=np.float32)

    for n in range(n_traj):
        x_0 = random_x0(intervals)
        U[n] = random_input(t, N_u, input_type, params=params)
        if input_type not in ("line", "circle", "figure8"):
            U[n, :, 1] *= 0.1
            U[n, :, 2] = 5 * np.abs(U[n, :, 2])
            U[n, :, -1] *= 0.05

        _, x = ct.input_output_response(rov_sys, t, U[n].T, x_0)
        x = x.T
        X[n, :, :3] = x[:, :3]
        X[n, :, 3] = np.cos(x[:, 3])
        X[n, :, 4] = np.sin(x[:, 3])
        X[n, :, 5:] = x[:, 4:]
        t_coll[n, :, 0] = dt

    os.makedirs(name, exist_ok=True)
    torch.save(torch.from_numpy(t), os.path.join(name, "t.pt"))
    torch.save(torch.from_numpy(U), os.path.join(name, "U.pt"))
    torch.save(torch.from_numpy(X), os.path.join(name, "X.pt"))
    torch.save(torch.from_numpy(t_coll), os.path.join(name, "t_coll.pt"))
    print(f"  {name}: {n_traj} trajectories, {N} steps each")
    return X, U, t

In [ ]:
# Generate datasets (reduce n_traj for quick notebook runs)
dt, T_tot = 0.08, 5.2

datasets = {
    "training_set": {
        "n_traj": 50,  # Use 400 for full training
        "input_type": "noise",
        "intervals": [1.0, 1.0, 1.0, np.pi, 1.0, 0.0, 0.1, 0.0],
    },
    "dev_set": {
        "n_traj": 20,  # Use 1000 for full training
        "input_type": "sine",
        "intervals": [0.0] * 8,
    },
    "test_set_interp": {
        "n_traj": 20,
        "input_type": "sine",
        "intervals": [0.0] * 8,
        "dt": dt - 0.02,
        "T_tot": 3.9,
    },
    "test_set_extrap": {
        "n_traj": 20,
        "input_type": "sine",
        "intervals": [0.0] * 8,
        "dt": dt + 0.02,
        "T_tot": 6.5,
    },
}

print("Generating datasets...")
generated = {}
for name, cfg in datasets.items():
    d = cfg.get("dt", dt)
    T = cfg.get("T_tot", T_tot)
    X, U, t = generate_dataset(name, cfg["n_traj"], cfg["input_type"], d, T, cfg["intervals"])
    generated[name] = (X, U, t)

print("\nDone!")

In [ ]:
# Visualise a few training trajectories
X_train, U_train, t_train = generated["training_set"]
state_labels = ["x", "y", "z", "cos(ψ)", "sin(ψ)", "u", "v", "w", "r"]

fig, axes = plt.subplots(3, 3, figsize=(14, 9), sharex=True)
for s, ax in enumerate(axes.flatten()):
    for j in range(min(5, X_train.shape[0])):
        ax.plot(t_train, X_train[j, :, s], alpha=0.7, linewidth=0.8)
    ax.set_ylabel(state_labels[s])
    ax.grid(True, alpha=0.3)
axes[-1, 1].set_xlabel("Time (s)")
fig.suptitle("Sample training trajectories", fontsize=14)
plt.tight_layout()
plt.show()

## 3. Graph Structure Inspection

Build a sample graph and inspect its topology — this is the core novelty of the PIGNN approach.

In [ ]:
from models.graph_builder import build_graph, allocate_thrusts, NUM_THRUSTERS

# Build a graph from the first time-step of the first training trajectory
sample_state = torch.tensor(X_train[0, 0], dtype=torch.float32)
sample_tau = torch.tensor(U_train[0, 0], dtype=torch.float32)

graph = build_graph(sample_state, sample_tau)

print("=== Heterogeneous Graph Structure ===")
print(f"\nNode types:")
for ntype in graph.node_types:
    x = graph[ntype].x
    print(f"  {ntype:20s}  nodes={x.shape[0]}, features={x.shape[1]}")

print(f"\nEdge types:")
for etype in graph.edge_types:
    ei = graph[etype].edge_index
    ea = graph[etype].edge_attr
    name = f"{etype[0]} --[{etype[1]}]--> {etype[2]}"
    print(f"  {name:45s}  edges={ei.shape[1]}, edge_feat_dim={ea.shape[1]}")

print(f"\nTotal nodes: {sum(graph[nt].x.shape[0] for nt in graph.node_types)}")
print(f"Total edges: {sum(graph[et].edge_index.shape[1] for et in graph.edge_types)}")

In [ ]:
# Verify hard constraint: all edges target hull
print("Hard constraint check — all edges target hull:")
for etype in graph.edge_types:
    target = etype[2]
    status = "✓" if target == "hull" else "✗ VIOLATION"
    print(f"  {etype[0]:15s} → {target:10s}  {status}")

# Inspect thruster allocation
print(f"\nThruster allocation for τ = {sample_tau.numpy()}:")
f_alloc = allocate_thrusts(sample_tau.unsqueeze(0)).squeeze()
for i in range(NUM_THRUSTERS):
    print(f"  Thruster {i}: f = {f_alloc[i].item():.4f} N")

## 4. Model & Training

Instantiate the PIGNN and train it with the physics-informed loss.

In [ ]:
from models.pignn import PIGNN
from models.model_utility import (
    get_data_sets,
    convert_input_data,
    convert_output_data,
    data_loss_fn,
    initial_condition_loss,
    physics_loss_fn,
    rollout_loss_fn,
)

# Quick sanity check: forward pass
model = PIGNN(N_in=14, N_out=9, hidden=32, msg_dim=32, n_mp_layers=2).to(device)

n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model parameters: {n_params:,}")

# Test forward pass
Z_test = torch.randn(4, 14, device=device)
out = model(Z_test)
print(f"Forward pass:  input {Z_test.shape} → output {out.shape}")
assert out.shape == (4, 9), "Output shape mismatch!"
print("✓ Forward pass OK")

In [ ]:
# --- Training configuration ---
# Reduce epochs for notebook demo; use 1200 for full training
EPOCHS = 50
BATCH_SIZE = 3
LR = 8e-3
PINN = True
ROLLOUT = False  # Disable rollout for faster demo; enable for full training

from torch.utils.data import DataLoader
from data.data_utility import TrajectoryDataset

train_ds = TrajectoryDataset("training_set")
dev_ds = TrajectoryDataset("dev_set")
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
dev_loader = DataLoader(dev_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Training:   {len(train_ds)} trajectories")
print(f"Validation: {len(dev_ds)} trajectories")

In [ ]:
from torch.optim import AdamW
from torch.optim.lr_scheduler import ReduceLROnPlateau
from tqdm.notebook import trange

model = PIGNN(N_in=14, N_out=9, hidden=32, msg_dim=32, n_mp_layers=2).to(device)
optimizer = AdamW(model.parameters(), lr=LR)
scheduler = ReduceLROnPlateau(optimizer, factor=0.5, patience=200, min_lr=1e-5)

# Training history
history = {
    "train_data": [], "train_phys": [], "train_ic": [],
    "dev_data": [], "lr": [],
}

best_dev_loss = float("inf")
best_state = None

for epoch in trange(EPOCHS, desc="Training"):
    model.train()
    ep_data, ep_phys, ep_ic, n_batch = 0, 0, 0, 0

    for X, U, t_coll, time in train_loader:
        X, U = X.to(device), U.to(device)
        time, t_coll = time.to(device), t_coll.to(device)
        n_batch += 1

        l_data = data_loss_fn(model, X, U, time, device)
        l_ic = initial_condition_loss(model, X, U, time, device)

        l_total = l_data + l_ic
        if PINN:
            l_phys = physics_loss_fn(model, X, U, t_coll, device)
            l_total += 0.5 * l_phys
            ep_phys += l_phys.item()

        optimizer.zero_grad()
        l_total.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=5.0)
        optimizer.step()

        ep_data += l_data.item()
        ep_ic += l_ic.item()

    n_batch = max(n_batch, 1)
    history["train_data"].append(ep_data / n_batch)
    history["train_phys"].append(ep_phys / n_batch)
    history["train_ic"].append(ep_ic / n_batch)

    # --- Validation ---
    model.eval()
    dev_loss = 0
    n_dev = 0
    with torch.no_grad():
        for X, U, t_coll, time in dev_loader:
            X, U = X.to(device), U.to(device)
            time = time.to(device)
            Z, B, T, Nx = convert_input_data(X, U, time)
            X_hat = model(Z.to(device))
            X_hat = convert_output_data(X_hat, B, T, Nx)
            dev_loss += torch.nn.functional.mse_loss(X_hat[:, :-1], X[:, 1:]).item()
            n_dev += 1
    dev_loss /= max(n_dev, 1)
    history["dev_data"].append(dev_loss)
    history["lr"].append(optimizer.param_groups[0]["lr"])

    scheduler.step(dev_loss)

    if dev_loss < best_dev_loss:
        best_dev_loss = dev_loss
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

print(f"\nBest dev loss: {best_dev_loss:.6f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].semilogy(history["train_data"], label="Train (data)", alpha=0.8)
axes[0].semilogy(history["dev_data"], label="Dev (data)", alpha=0.8)
axes[0].set_xlabel("Epoch")
axes[0].set_ylabel("MSE (log)")
axes[0].set_title("Data Loss")
axes[0].legend()
axes[0].grid(True, alpha=0.3)

if PINN:
    axes[1].semilogy(history["train_phys"], label="Physics", color="C2", alpha=0.8)
    axes[1].set_xlabel("Epoch")
    axes[1].set_ylabel("MSE (log)")
    axes[1].set_title("Physics Loss")
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)
else:
    axes[1].text(0.5, 0.5, "PINN disabled", ha="center", va="center", transform=axes[1].transAxes)

axes[2].plot(history["lr"], color="C3")
axes[2].set_xlabel("Epoch")
axes[2].set_ylabel("Learning rate")
axes[2].set_title("Learning Rate Schedule")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 5. Evaluation — Rollout Predictions

Load the best model and run autoregressive rollouts on the dev set.

In [ ]:
# Load best model
model.load_state_dict(best_state)
model.eval()
model = model.to(device)


def rollout(model, X_gt, U, time):
    """Autoregressive rollout from the first time-step."""
    N_seq, Nx = X_gt.shape[1], X_gt.shape[2]
    X_pred = torch.zeros(N_seq, Nx, device=device)
    X_pred[0] = X_gt[0, 0]

    with torch.no_grad():
        for t in range(N_seq - 1):
            x_cur = X_pred[t].unsqueeze(0).unsqueeze(0)
            u_cur = U[:, t:t+1, :]
            t_cur = time[:, t:t+1, :]
            Z, B, T, Nx_ = convert_input_data(x_cur, u_cur, t_cur)
            x_next = model(Z.to(device))
            X_pred[t + 1] = x_next.squeeze()
    return X_pred.cpu().numpy()

In [ ]:
# Evaluate on a few dev-set trajectories
n_eval = min(4, len(dev_ds))
indices = np.random.choice(len(dev_ds), n_eval, replace=False)

fig, all_axes = plt.subplots(n_eval, 3, figsize=(15, 4 * n_eval), sharex=True)
if n_eval == 1:
    all_axes = all_axes[np.newaxis, :]

plot_states = [0, 2, 5]  # x, z, u (surge position, heave, surge velocity)
plot_labels = ["x (surge)", "z (heave)", "u (surge vel)"]

for row, idx in enumerate(indices):
    X_gt, U_gt, tc, tm = dev_ds[idx]
    X_gt_b = X_gt.unsqueeze(0).to(device)
    U_b = U_gt.unsqueeze(0).to(device)
    tm_b = tm.unsqueeze(0).to(device)

    X_pred = rollout(model, X_gt_b, U_b, tm_b)
    X_gt_np = X_gt.numpy()
    t_axis = tm[:, 0].numpy()

    for col, (si, sl) in enumerate(zip(plot_states, plot_labels)):
        ax = all_axes[row, col]
        ax.plot(t_axis, X_gt_np[:, si], "k-", linewidth=1.2, label="Ground truth")
        ax.plot(t_axis, X_pred[:, si], "r--", linewidth=1.0, label="PIGNN")
        ax.set_ylabel(sl)
        ax.grid(True, alpha=0.3)
        if row == 0:
            ax.set_title(sl)
        if col == 0:
            ax.legend(fontsize=8)

all_axes[-1, 1].set_xlabel("Time (s)")
fig.suptitle("Autoregressive rollout: PIGNN vs Ground Truth", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Per-state error summary
all_errors = []
for idx in range(min(20, len(dev_ds))):
    X_gt, U_gt, tc, tm = dev_ds[idx]
    X_gt_b = X_gt.unsqueeze(0).to(device)
    U_b = U_gt.unsqueeze(0).to(device)
    tm_b = tm.unsqueeze(0).to(device)
    X_pred = rollout(model, X_gt_b, U_b, tm_b)
    all_errors.append(np.abs(X_pred - X_gt.numpy()))

all_errors = np.stack(all_errors)  # (N, T, 9)
mean_err = all_errors.mean(axis=(0, 1))

print("Mean Absolute Error per state variable (rollout):")
print("-" * 40)
for s, label in enumerate(state_labels):
    print(f"  {label:>8s}: {mean_err[s]:.6f}")
print(f"{'Overall':>10s}: {mean_err.mean():.6f}")

In [ ]:
# Save the best model
os.makedirs("models_saved", exist_ok=True)
save_path = "models_saved/pignn_notebook_best.pt"
torch.save(best_state, save_path)
print(f"Model saved to {save_path}")

## 6. Graph Visualisation

Visualise the heterogeneous graph topology using matplotlib.

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(10, 8))

# Node positions for visualisation
positions = {
    "hull": (0.5, 0.5),
    "hydrodynamic": (0.1, 0.85),
    "buoyancy": (0.9, 0.85),
}
# Thrusters arranged in a semicircle below
for i in range(8):
    angle = np.pi * (0.15 + 0.7 * i / 7)
    positions[f"thruster_{i}"] = (0.5 + 0.38 * np.cos(angle), 0.5 - 0.35 * np.sin(angle))

colors = {
    "hull": "#2196F3",
    "thruster": "#FF9800",
    "hydrodynamic": "#4CAF50",
    "buoyancy": "#9C27B0",
}

# Draw edges
for i in range(8):
    tx, ty = positions[f"thruster_{i}"]
    hx, hy = positions["hull"]
    ax.annotate("", xy=(hx, hy), xytext=(tx, ty),
                arrowprops=dict(arrowstyle="->", color=colors["thruster"], lw=1.5, alpha=0.6))

for src in ["hydrodynamic", "buoyancy"]:
    sx, sy = positions[src]
    hx, hy = positions["hull"]
    ax.annotate("", xy=(hx, hy), xytext=(sx, sy),
                arrowprops=dict(arrowstyle="->", color=colors[src], lw=2.5))

# Draw nodes
ax.scatter(*positions["hull"], s=1200, c=colors["hull"], zorder=5, edgecolors="white", linewidth=2)
ax.text(*positions["hull"], "Hull\n(1 node)", ha="center", va="center", fontsize=9, fontweight="bold", color="white")

for i in range(8):
    px, py = positions[f"thruster_{i}"]
    ax.scatter(px, py, s=400, c=colors["thruster"], zorder=5, edgecolors="white", linewidth=1.5)
    ax.text(px, py, f"T{i}", ha="center", va="center", fontsize=7, fontweight="bold", color="white")

for ntype in ["hydrodynamic", "buoyancy"]:
    px, py = positions[ntype]
    ax.scatter(px, py, s=800, c=colors[ntype], zorder=5, edgecolors="white", linewidth=2)
    label = "Hydro\n(1 node)" if ntype == "hydrodynamic" else "Buoyancy\n(1 node)"
    ax.text(px, py, label, ha="center", va="center", fontsize=8, fontweight="bold", color="white")

# Legend
from matplotlib.patches import FancyArrowPatch
legend_elements = [
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors["hull"], markersize=12, label="Hull (state)"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors["thruster"], markersize=10, label="Thruster × 8"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors["hydrodynamic"], markersize=10, label="Hydrodynamic"),
    plt.Line2D([0], [0], marker="o", color="w", markerfacecolor=colors["buoyancy"], markersize=10, label="Buoyancy"),
]
ax.legend(handles=legend_elements, loc="upper center", fontsize=9, ncol=4)

ax.set_xlim(-0.05, 1.05)
ax.set_ylim(-0.05, 1.05)
ax.set_aspect("equal")
ax.axis("off")
ax.set_title("BlueROV2 PIGNN — Heterogeneous Graph Topology", fontsize=14, pad=20)
plt.tight_layout()
plt.show()

---

## Summary

This notebook demonstrated the full PIGNN pipeline:

- **Graph structure** encodes physical priors as hard constraints (only thrusters introduce forces; all edges target hull)
- **Message passing** aggregates forces from thrusters (sum), drag, and buoyancy into the hull node
- **Physics-informed loss** penalises deviations from the 4-DOF Fossen equation
- **Residual prediction** with body→world rotation for position increments

For full-scale training, increase `n_traj` in the data generation step and set `EPOCHS = 1200`, `ROLLOUT = True`.